# Q1. Embedding a query

In [115]:
import numpy as np
from embedder import Embedder

In [116]:
embed = Embedder()

In [117]:
q1 = "How does approximate nearest neighbor search work?"
v = embed.encode(q1)
v[0]

np.float64(-0.02058203437252893)

## Loading the data

In [118]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

# Q2. Cosine similarity

In [119]:
q2 = next(d for d in documents if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md")

In [120]:
dv = embed.encode(q2["content"])

In [121]:
v.dot(dv)

np.float64(0.36107026789538205)

# Q3. Chunking and search by hand

In [122]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [123]:
X = embed.encode_batch([chunk["content"] for chunk in chunks])
X = np.array(X)

In [124]:
scores = X.dot(v)

best_idx = np.argmax(scores)
best_chunk = chunks[best_idx]

In [125]:
best_chunk['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

# Q4. Vector search with minsearch

In [126]:
from minsearch import Index, VectorSearch

In [127]:
vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [128]:
q4 = "What metric do we use to evaluate a search engine?"
v_q4 = embed.encode(q4)

In [129]:
results_q4 = vindex.search(v_q4, num_results=1)
results_q4[0]

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

# Q5. Text search vs vector search

In [130]:
q5 = "How do I store vectors in PostgreSQL?"

In [131]:
v_q5 = embed.encode(q5)
vector_results = vindex.search(v_q5, num_results=5)
vector_filenames = [r["filename"] for r in vector_results]

In [132]:
tindex = Index(text_fields=["content"], keyword_fields=["filename"])
tindex.fit(chunks)
text_results = tindex.search(q5, num_results=5)
text_filenames = [r["filename"] for r in text_results]

In [133]:
only_in_vector = set(vector_filenames) - set(text_filenames)
only_in_vector

{'02-vector-search/lessons/08-pgvector.md'}

# Q6. Hybrid search

In [134]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [135]:
q6 = "How do I give the model access to tools?"

In [136]:
v_q6 = embed.encode(q6)
v_res_q6 = vindex.search(v_q6, num_results=5)

In [137]:
t_res_q6 = tindex.search(q6, num_results=5)

In [138]:
hybrid_results = rrf([v_res_q6, t_res_q6])
hybrid_results[0]['filename']

'01-agentic-rag/lessons/13-function-calling.md'